# 🏷️ 2.3 — Codificación de Variables Categóricas
## Diplomado en Ciencia Actuarial y Analítica de Seguros

**Objetivo:** Convertir variables de texto a numéricas para modelos predictivos.  
**Variables típicas en seguros:** Tipo de cobertura, municipio, marca de auto, profesión, sexo.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print("✅ Librerías listas")


In [ ]:
# Dataset de pólizas de seguro de auto
np.random.seed(123)
n = 800

municipios = ['Monterrey', 'San Pedro', 'Guadalupe', 'Apodaca', 'Escobedo', 
              'Santa Catarina', 'Juárez', 'García']
coberturas = ['Básica', 'Amplia', 'Total']
niveles_riesgo = ['Bajo', 'Medio', 'Alto']  # Variable ordinal
marcas = ['Toyota', 'Nissan', 'Chevrolet', 'Honda', 'Ford', 'Volkswagen', 
          'Hyundai', 'Kia', 'Mazda', 'Seat']

df = pd.DataFrame({
    'id_poliza': range(1, n+1),
    'municipio': np.random.choice(municipios, n, p=[0.25, 0.15, 0.18, 0.12, 0.08, 0.08, 0.07, 0.07]),
    'tipo_cobertura': np.random.choice(coberturas, n, p=[0.40, 0.35, 0.25]),
    'nivel_riesgo': np.random.choice(niveles_riesgo, n, p=[0.45, 0.35, 0.20]),
    'marca_auto': np.random.choice(marcas, n),
    'sexo': np.random.choice(['M', 'F'], n, p=[0.58, 0.42]),
    'prima_anual': np.random.lognormal(9.8, 0.5, n).clip(5000, 100000),
    'siniestros': np.random.poisson(0.25, n),
})

print(f"📋 Dataset: {df.shape[0]} pólizas, {df.shape[1]} columnas")
print(df.head(8).to_string(index=False))
print(f"\n🔢 Tipos de datos:")
print(df.dtypes)


## 1. One-Hot Encoding (OHE)

Crea una columna binaria (0/1) por cada categoría.  
⚠️ **Regla:** Se elimina una columna de referencia para evitar la "trampa de la dummy".


In [ ]:
# One-Hot Encoding con pandas
print("ANTES:")
print(df[['id_poliza', 'tipo_cobertura']].head(8))

# Método 1: pd.get_dummies (el más sencillo)
dummies_cobertura = pd.get_dummies(df['tipo_cobertura'], prefix='cobertura', drop_first=True)
print("\nDESPUÉS (con drop_first=True — elimina categoría de referencia 'Amplia'):")
print(dummies_cobertura.head(8).astype(int))

# Agregar al dataframe
df_ohe = pd.concat([df, dummies_cobertura], axis=1)
print(f"\n✅ Se crearon {dummies_cobertura.shape[1]} columnas nuevas")
print("   Columna de referencia eliminada: 'cobertura_Amplia'")
print("   Interpretación: cobertura_Básica=0, cobertura_Total=0 → es Amplia")


In [ ]:
# OHE para múltiples variables categóricas
vars_categoricas = ['municipio', 'tipo_cobertura', 'sexo']

df_encoded = pd.get_dummies(df, columns=vars_categoricas, drop_first=True, dtype=int)

print(f"Columnas originales: {df.shape[1]}")
print(f"Columnas después de OHE: {df_encoded.shape[1]}")
print(f"Columnas nuevas creadas: {df_encoded.shape[1] - df.shape[1]}")
print("\nNuevas columnas creadas:")
nuevas = [c for c in df_encoded.columns if any(v in c for v in vars_categoricas)]
for c in nuevas:
    print(f"  • {c}")


In [ ]:
# Visualizar frecuencias de categorías
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
categoricas = ['municipio', 'tipo_cobertura', 'sexo']
colors = ['#0A3D62', '#1ABC9C', '#F39C12']

for ax, col, c in zip(axes, categoricas, colors):
    freq = df[col].value_counts()
    bars = ax.barh(freq.index, freq.values, color=c, alpha=0.85)
    ax.set_title(f'Distribución: {col}', fontweight='bold')
    ax.set_xlabel('Frecuencia')
    for bar, v in zip(bars, freq.values):
        ax.text(v + 2, bar.get_y() + bar.get_height()/2, str(v), va='center', fontsize=9)

plt.suptitle('Frecuencia de Variables Categóricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('categoricas_freq.png', dpi=120, bbox_inches='tight')
plt.show()


## 2. Label Encoding — Para Variables Ordinales

Solo usar cuando las categorías tienen un **orden natural**.  
Ejemplo: Nivel de riesgo (Bajo < Medio < Alto).


In [ ]:
# Label Encoding CORRECTO: variables con orden natural
orden_riesgo = {'Bajo': 0, 'Medio': 1, 'Alto': 2}
df['nivel_riesgo_encoded'] = df['nivel_riesgo'].map(orden_riesgo)

print("Variable ordinal con order correcto:")
print(df[['nivel_riesgo', 'nivel_riesgo_encoded']].drop_duplicates().sort_values('nivel_riesgo_encoded'))

# Verificar que el orden tiene sentido actuarial
print("\n📊 Prima promedio por nivel de riesgo:")
print(df.groupby('nivel_riesgo')['prima_anual'].mean().reindex(['Bajo', 'Medio', 'Alto']).apply(lambda x: f'${x:,.0f}'))

# Label Encoding INCORRECTO: variables sin orden
le = LabelEncoder()
df['municipio_le_incorrecto'] = le.fit_transform(df['municipio'])
print("\n⚠️  Label Encoding INCORRECTO en municipio (sin orden natural):")
for mun, code in zip(le.classes_, range(len(le.classes_))):
    print(f"   {mun} → {code}  ← ¿tiene sentido que Apodaca=0 < García=3?")


## 3. Target Encoding — Para Alta Cardinalidad

Reemplaza cada categoría con el **promedio de la variable objetivo** para esa categoría.  
⚠️ Solo calcular con datos de entrenamiento para evitar Data Leakage.


In [ ]:
# Target Encoding: municipio → tasa de siniestralidad
# Calcular tasa de siniestralidad por municipio
df['tiene_siniestro'] = (df['siniestros'] > 0).astype(int)

target_encoding = df.groupby('municipio')['tiene_siniestro'].agg(['mean', 'count'])
target_encoding.columns = ['tasa_siniestralidad', 'num_polizas']
target_encoding = target_encoding.sort_values('tasa_siniestralidad', ascending=False)

print("📊 Target Encoding por Municipio:")
print(target_encoding.round(4))

# Aplicar al dataframe
mapa_encoding = target_encoding['tasa_siniestralidad'].to_dict()
df['municipio_target_encoded'] = df['municipio'].map(mapa_encoding)

print(f"\n✅ Municipio codificado con tasa de siniestralidad")
print(df[['municipio', 'municipio_target_encoded']].drop_duplicates().sort_values('municipio_target_encoded', ascending=False))


In [ ]:
# Target Encoding con suavizado (Smoothing) — más robusto
# Fórmula: θ = (n_cat * media_cat + k * media_global) / (n_cat + k)
# k = parámetro de suavizado (típico: 10-30)

media_global = df['tiene_siniestro'].mean()
k = 20  # parámetro de suavizado

target_smooth = df.groupby('municipio')['tiene_siniestro'].agg(['mean', 'count'])
target_smooth['encoding_suavizado'] = (
    (target_smooth['count'] * target_smooth['mean'] + k * media_global) /
    (target_smooth['count'] + k)
)

print(f"Media global de siniestros: {media_global:.4f}")
print("\nComparación: Target Encoding vs Suavizado:")
print(target_smooth[['mean', 'count', 'encoding_suavizado']].rename(
    columns={'mean': 'Media_categoria', 'count': 'N_polizas', 'encoding_suavizado': 'Suavizado'}
).round(4))
print("\n💡 Con pocos datos, el suavizado acerca los valores a la media global")


In [ ]:
# Tabla resumen final
print("="*65)
print("GUÍA DE SELECCIÓN: ¿Cuál encoding usar?")
print("="*65)
guia = pd.DataFrame({
    'Técnica': ['One-Hot Encoding', 'Label Encoding', 'Target Encoding'],
    'Cuándo': ['Pocas categorías (<20), sin orden', 'Ordinal: Bajo/Medio/Alto', 'Muchas categorías (código postal, municipio)'],
    'Ejemplo': ['Tipo cobertura, Sexo', 'Nivel riesgo, Rango edad', 'Municipio, Marca auto'],
    'Riesgo': ['Trampa dummy', 'Orden falso si no es ordinal', 'Data leakage'],
})
print(guia.to_string(index=False))


# 📅 2.4 — Variables Dummy y Derivadas Actuariales
## Diplomado en Ciencia Actuarial y Analítica de Seguros

**Objetivo:** Crear variables actuariales fundamentales: edad, exposición y antigüedad.  
**Importancia:** Estas variables son la base de la tarificación y análisis de riesgo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print("✅ Librerías cargadas")

In [ ]:
# Crear dataset de pólizas con fechas
np.random.seed(42)
n = 600
fecha_base = pd.Timestamp('2020-01-01')
fecha_actual = pd.Timestamp('2024-12-31')

# Generar fechas aleatorias
def fecha_aleatoria(inicio, fin, n):
    rango = (fin - inicio).days
    dias = np.random.randint(0, rango, n)
    return pd.to_datetime([inicio + timedelta(days=int(d)) for d in dias])

fechas_nacimiento = fecha_aleatoria(pd.Timestamp('1955-01-01'), pd.Timestamp('2005-12-31'), n)
fechas_inicio_poliza = fecha_aleatoria(pd.Timestamp('2015-01-01'), pd.Timestamp('2023-01-01'), n)
duración_dias = np.random.choice([365, 180, 90, 730], n, p=[0.55, 0.20, 0.10, 0.15])
fechas_fin_poliza = fechas_inicio_poliza + pd.to_timedelta(duración_dias, unit='d')
fechas_primer_poliza = fechas_inicio_poliza - pd.to_timedelta(np.random.randint(0, 3650, n), unit='d')

df = pd.DataFrame({
    'id_asegurado': range(1, n+1),
    'fecha_nacimiento': fechas_nacimiento,
    'fecha_inicio_poliza': fechas_inicio_poliza,
    'fecha_fin_poliza': fechas_fin_poliza.clip(upper=fecha_actual),
    'fecha_primera_poliza': fechas_primer_poliza,
    'sexo': np.random.choice(['M', 'F'], n),
    'num_siniestros': np.random.poisson(0.2, n),
    'monto_siniestros': np.random.lognormal(8, 1, n) * np.random.poisson(0.2, n),
})

print(f"📋 Dataset: {df.shape}")
print(df[['id_asegurado', 'fecha_nacimiento', 'fecha_inicio_poliza', 'fecha_fin_poliza']].head(6))

## 1. Cálculo de Edad

La edad **no** se debe guardar como un número fijo — cambia con el tiempo.  
En actuaría calculamos la edad al momento de evaluación o siniestro.

In [ ]:
# Calcular edad correctamente
FECHA_EVALUACION = pd.Timestamp('2024-12-31')

# Edad en años decimales (más preciso)
df['edad_anios'] = (FECHA_EVALUACION - df['fecha_nacimiento']).dt.days / 365.25

# Edad en años completos
df['edad_cumplida'] = df['edad_anios'].astype(int)

# Verificar
print("📊 Edad calculada:")
print(df[['fecha_nacimiento', 'edad_anios', 'edad_cumplida']].head(8).to_string(index=False))
print(f"\n📈 Distribución de edades:")
print(f"  Mínima: {df['edad_cumplida'].min()} años")
print(f"  Máxima: {df['edad_cumplida'].max()} años")
print(f"  Promedio: {df['edad_anios'].mean():.1f} años")

# Segmentación por grupos de edad (estándar actuarial)
bins_edad = [0, 25, 35, 45, 55, 65, 200]
labels_edad = ['<25', '25-34', '35-44', '45-54', '55-64', '65+']
df['grupo_edad'] = pd.cut(df['edad_cumplida'], bins=bins_edad, labels=labels_edad, right=False)
print("\n👥 Distribución por grupo de edad:")
print(df['grupo_edad'].value_counts().sort_index())

## 2. Cálculo de Exposición

La **exposición** mide el tiempo real que el asegurado estuvo bajo cobertura.  
Es fundamental para calcular tasas de siniestralidad comparables.

In [ ]:
# Calcular exposición (tiempo bajo cobertura)
# Exposición en días
df['exposicion_dias'] = (df['fecha_fin_poliza'] - df['fecha_inicio_poliza']).dt.days.clip(lower=0)

# Exposición en años (estándar actuarial)
df['exposicion_anios'] = df['exposicion_dias'] / 365.25

# Frecuencia ajustada por exposición
df['frecuencia_siniestros'] = np.where(
    df['exposicion_anios'] > 0,
    df['num_siniestros'] / df['exposicion_anios'],
    0
)

print("📊 Variables de exposición:")
print(df[['id_asegurado', 'fecha_inicio_poliza', 'fecha_fin_poliza', 
          'exposicion_dias', 'exposicion_anios', 'num_siniestros', 'frecuencia_siniestros']].head(8).to_string(index=False))

print(f"\n📈 Estadísticas de exposición:")
print(f"  Exposición total del portafolio: {df['exposicion_anios'].sum():.1f} póliza-años")
print(f"  Promedio por póliza: {df['exposicion_anios'].mean():.3f} años ({df['exposicion_dias'].mean():.0f} días)")
print(f"  Frecuencia media: {df['frecuencia_siniestros'].mean():.4f} siniestros/año")

## 3. Antigüedad del Cliente

La antigüedad mide la **lealtad** del asegurado. Clientes más antiguos suelen tener mejor comportamiento.

In [ ]:
# Calcular antigüedad
df['antiguedad_anios'] = (FECHA_EVALUACION - df['fecha_primera_poliza']).dt.days / 365.25
df['antiguedad_anios'] = df['antiguedad_anios'].clip(lower=0)

# Segmentación por antigüedad
bins_ant = [0, 1, 3, 5, 10, 100]
labels_ant = ['<1 año', '1-3 años', '3-5 años', '5-10 años', '10+ años']
df['segmento_antiguedad'] = pd.cut(df['antiguedad_anios'], bins=bins_ant, labels=labels_ant)

# Análisis de siniestralidad por antigüedad
analisis_ant = df.groupby('segmento_antiguedad', observed=True).agg(
    n_polizas=('id_asegurado', 'count'),
    siniestros_total=('num_siniestros', 'sum'),
    exposicion_total=('exposicion_anios', 'sum'),
    prima_media=('monto_siniestros', 'mean')
).round(3)
analisis_ant['frecuencia'] = (analisis_ant['siniestros_total'] / analisis_ant['exposicion_total']).round(4)

print("📊 Siniestralidad por Antigüedad:")
print(analisis_ant.to_string())
print("\n💡 Observa si los clientes más antiguos tienen menor frecuencia de siniestros")

In [ ]:
# Variables Dummy específicas actuariales
df['es_renovacion'] = (df['antiguedad_anios'] > 1).astype(int)
df['tiene_historial_siniestros'] = (df['num_siniestros'] > 0).astype(int)
df['poliza_anual'] = (df['exposicion_dias'] >= 360).astype(int)
df['adulto_mayor'] = (df['edad_cumplida'] >= 65).astype(int)
df['conductor_joven'] = (df['edad_cumplida'] < 26).astype(int)
# NCD Score (No Claims Discount)
anos_sin_siniestro = np.where(df['num_siniestros'] == 0, df['antiguedad_anios'].clip(0, 5), 0)
df['ncd_score'] = anos_sin_siniestro * 0.05  # 5% de descuento por año sin siniestro

print("📋 Variables dummy y derivadas creadas:")
print(df[['id_asegurado', 'es_renovacion', 'tiene_historial_siniestros', 
          'poliza_anual', 'adulto_mayor', 'conductor_joven', 'ncd_score']].head(10).to_string(index=False))

print("\n📊 Prevalencia de cada dummy:")
for col in ['es_renovacion', 'tiene_historial_siniestros', 'poliza_anual', 'adulto_mayor', 'conductor_joven']:
    pct = df[col].mean() * 100
    print(f"  {col}: {df[col].sum()} ({pct:.1f}%)")

In [ ]:
# Visualización de variables derivadas
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Distribución de edad
axes[0,0].hist(df['edad_cumplida'], bins=30, color='#0A3D62', alpha=0.8, edgecolor='white')
axes[0,0].set_title('Distribución de Edad', fontweight='bold')
axes[0,0].set_xlabel('Edad (años)')
axes[0,0].axvline(df['edad_cumplida'].mean(), color='red', ls='--', label=f'Media: {df['edad_cumplida'].mean():.1f}')
axes[0,0].legend()

# Distribución de exposición
axes[0,1].hist(df['exposicion_anios'], bins=25, color='#1ABC9C', alpha=0.8, edgecolor='white')
axes[0,1].set_title('Distribución de Exposición', fontweight='bold')
axes[0,1].set_xlabel('Exposición (años)')

# Distribución de antigüedad
axes[0,2].hist(df['antiguedad_anios'], bins=25, color='#F39C12', alpha=0.8, edgecolor='white')
axes[0,2].set_title('Distribución de Antigüedad', fontweight='bold')
axes[0,2].set_xlabel('Antigüedad (años)')

# Siniestros por grupo de edad
freq_edad = df.groupby('grupo_edad', observed=True)['frecuencia_siniestros'].mean()
axes[1,0].bar(freq_edad.index, freq_edad.values, color='#0A3D62', alpha=0.8)
axes[1,0].set_title('Frecuencia por Grupo de Edad', fontweight='bold')
axes[1,0].set_xlabel('Grupo de edad')
axes[1,0].set_ylabel('Frecuencia media (sin/año)')

# Siniestros por antigüedad
freq_ant = df.groupby('segmento_antiguedad', observed=True)['frecuencia_siniestros'].mean()
axes[1,1].bar(freq_ant.index, freq_ant.values, color='#8E44AD', alpha=0.8)
axes[1,1].set_title('Frecuencia por Antigüedad', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=30)

# NCD Score distribution
axes[1,2].hist(df['ncd_score'], bins=20, color='#27AE60', alpha=0.8, edgecolor='white')
axes[1,2].set_title('Distribución NCD Score', fontweight='bold')
axes[1,2].set_xlabel('Descuento (0 = sin descuento, 0.25 = 25%)')

plt.suptitle('Variables Actuariales Derivadas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('variables_actuariales.png', dpi=120, bbox_inches='tight')
plt.show()